# 43008 Assignment 2 - Breakout DQN and PPO (AMD DirectML GPU)

**Student:** replace name and ID  
**Environment:** Gymnasium/ALE `Breakout`  
**Algorithms:** DQN and PPO  
**Custom extractor:** student-defined CNN

This notebook has intentionally not been executed. Run it top-to-bottom and retain all outputs. Each algorithm targets at least 1,000,000 cumulative timesteps and must be evaluated against a random baseline.

## 1. Setup and reproducibility

CartPole has a four-value continuous observation and two discrete actions. Breakout uses stacked image frames and a discrete action set. DQN learns action values with replay and a target network; PPO directly optimises a clipped actor-critic objective.

In [ ]:
# Run once, then restart the kernel.
%pip install "gymnasium[classic-control,atari,accept-rom-license]" "stable-baselines3[extra]" ale-py autorom moviepy imageio-ffmpeg pandas matplotlib torch-directml

In [ ]:
from pathlib import Path
import json, os, random, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import ale_py
import torch as th
import torch.nn as nn
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecVideoRecorder
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.evaluation import evaluate_policy

gym.register_envs(ale_py)
SEED = 2026
random.seed(SEED); np.random.seed(SEED); th.manual_seed(SEED)
ROOT = Path.cwd()
OUT = ROOT / "remaining_artifacts" / "gpu_amd_directml" / "breakout"
for p in [OUT/"models", OUT/"logs", OUT/"figures", OUT/"videos"]: p.mkdir(parents=True, exist_ok=True)
print("Python/PyTorch/Gymnasium:", th.__version__, gym.__version__)

In [ ]:
import torch_directml
DEVICE = torch_directml.device()
if "privateuseone" not in str(DEVICE):
    raise RuntimeError("DirectML device was not created; do not continue on CPU silently.")
print("DirectML device:", DEVICE)

## 2. Custom feature extractors

These are student-defined extractors passed through `policy_kwargs`, satisfying the custom ANN/CNN requirement. Network diagrams: `4 -> 128 -> 128 -> 128 features` for CartPole; `stacked frames -> Conv(32) -> Conv(64) -> Conv(64) -> 512 features` for Breakout.

In [ ]:
class CustomANN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=128):
        super().__init__(observation_space, features_dim)
        n = int(np.prod(observation_space.shape))
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(n,128), nn.ReLU(), nn.Linear(128,128), nn.ReLU(), nn.Linear(128,features_dim), nn.ReLU())
    def forward(self, observations): return self.net(observations.float())

class CustomCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=512):
        super().__init__(observation_space, features_dim)
        c = observation_space.shape[0]
        self.cnn = nn.Sequential(nn.Conv2d(c,32,8,4),nn.ReLU(),nn.Conv2d(32,64,4,2),nn.ReLU(),nn.Conv2d(64,64,3,1),nn.ReLU(),nn.Flatten())
        with th.no_grad(): flat = self.cnn(th.as_tensor(observation_space.sample()[None]).float()/255.0).shape[1]
        self.linear = nn.Sequential(nn.Linear(flat,features_dim),nn.ReLU())
    def forward(self, observations): return self.linear(self.cnn(observations.float()/255.0))

ANN_KW = dict(features_extractor_class=CustomANN, features_extractor_kwargs=dict(features_dim=128))
CNN_KW = dict(features_extractor_class=CustomCNN, features_extractor_kwargs=dict(features_dim=512), normalize_images=False)

## 3. Checkpoint callback

Models save periodically and resume toward a fixed target. DQN replay-buffer saving is enabled; it can be large but preserves genuine off-policy continuation.

In [ ]:
class SaveProgress(BaseCallback):
    def __init__(self, algorithm, folder, save_freq, save_replay=False):
        super().__init__(); self.algorithm=algorithm; self.folder=Path(folder); self.save_freq=save_freq; self.save_replay=save_replay; self.next_save=save_freq
    def _on_step(self):
        if self.num_timesteps >= self.next_save:
            self.model.save(self.folder/f"{self.algorithm}_latest")
            if self.save_replay and hasattr(self.model,"save_replay_buffer"): self.model.save_replay_buffer(self.folder/f"{self.algorithm}_replay.pkl")
            self.next_save += self.save_freq
        return True

def load_or_create(cls, path, env, kwargs):
    if Path(str(path)+".zip").exists():
        model=cls.load(path, env=env, device=DEVICE)
        replay=Path(str(path).replace("_latest", "_replay.pkl"))
        if cls is DQN and replay.exists(): model.load_replay_buffer(replay)
        print("Resumed", path, "at timestep", model.num_timesteps)
        return model
    return cls(env=env, device=DEVICE, verbose=1, seed=SEED, **kwargs)

## 5. Breakout environment and preprocessing

Training uses 84x84 grayscale observations, action repeat/max-pooling, frame stacking and reward clipping. Evaluation disables reward clipping so reported scores remain in the game's original reward scale.

In [ ]:
def make_breakout(training=True):
    def factory():
        env=gym.make("ALE/Breakout-v5",frameskip=1,repeat_action_probability=0.0)
        env=AtariWrapper(env,screen_size=84,frame_skip=4,terminal_on_life_loss=False,clip_reward=training)
        return Monitor(env)
    return VecFrameStack(DummyVecEnv([factory]),n_stack=4,channels_order="last")

test_env=make_breakout(False)
print("Breakout observation/action:",test_env.observation_space,test_env.action_space)
test_env.close()

In [ ]:
def breakout_random_baseline(episodes=30):
    env = make_breakout(False)
    returns = []
    for episode in range(episodes):
        obs = env.reset(); done = np.array([False]); total = 0.0
        while not done[0]:
            action = np.array([env.action_space.sample()])
            obs, reward, done, info = env.step(action)
            total += float(reward[0])
        returns.append(total)
    env.close()
    returns = np.asarray(returns)
    print(f"Random Breakout mean={returns.mean():.2f}, std={returns.std(ddof=1):.2f}")
    return returns

breakout_random_returns = breakout_random_baseline(30)

## 6. Breakout DQN and PPO training

Both targets are fixed at 1,000,000 cumulative timesteps. Re-running resumes only the missing difference. Increase the target if the agent does not beat the random baseline.

In [ ]:
ATARI_TARGET_TIMESTEPS=1_000_000
atari_results=[]
for name,cls,kwargs in [
    ("breakout_dqn",DQN,dict(policy="CnnPolicy",policy_kwargs=CNN_KW,learning_rate=1e-4,buffer_size=50000,learning_starts=50000,batch_size=32,gamma=.99,train_freq=4,target_update_interval=10000,exploration_fraction=.1,exploration_final_eps=.01,optimize_memory_usage=True)),
    ("breakout_ppo",PPO,dict(policy="CnnPolicy",policy_kwargs=CNN_KW,learning_rate=2.5e-4,n_steps=128,batch_size=256,n_epochs=4,gamma=.99,gae_lambda=.95,clip_range=.1,ent_coef=.01)),
]:
    env=make_breakout(True); latest=OUT/"models"/f"{name}_latest"; model=load_or_create(cls,latest,env,kwargs)
    remaining=max(0,ATARI_TARGET_TIMESTEPS-model.num_timesteps)
    if remaining: model.learn(total_timesteps=remaining,callback=SaveProgress(name,OUT/"models",100000,save_replay=(cls is DQN)),reset_num_timesteps=False)
    model.save(latest)
    if cls is DQN: model.save_replay_buffer(OUT/"models"/f"{name}_replay.pkl")
    eval_env=make_breakout(False); mean,std=evaluate_policy(model,eval_env,n_eval_episodes=30,deterministic=True)
    atari_results.append(dict(policy=name,mean_return=mean,std_return=std,timesteps=model.num_timesteps))
    env.close(); eval_env.close()
display(pd.DataFrame(atari_results))

## 7. Curves, videos and submission outputs

Use Monitor CSV files to plot episode reward and a 100-episode rolling mean. Record one deterministic episode for each final model and report mean ± standard deviation against the random baseline. Retain model ZIPs, replay buffer, CSV logs, figures and MP4 files.

In [ ]:
def record_breakout(model_path, algorithm_class, name, max_steps=10000):
    env = make_breakout(False)
    model = algorithm_class.load(model_path, env=env, device=DEVICE)
    video = VecVideoRecorder(
        env, str(OUT/"videos"), record_video_trigger=lambda step: step == 0,
        video_length=max_steps, name_prefix=name,
    )
    obs = video.reset()
    for _ in range(max_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = video.step(action)
        if done[0]: break
    video.close()

# Run after both training cells have completed:
# record_breakout(OUT/"models"/"breakout_dqn_latest", DQN, "breakout-dqn")
# record_breakout(OUT/"models"/"breakout_ppo_latest", PPO, "breakout-ppo")

pd.DataFrame(atari_results).to_csv(OUT/"breakout_evaluation.csv", index=False)
print("Evaluation table saved to", OUT/"breakout_evaluation.csv")

## 8. Report discussion checklist

- State/action/reward/termination for both environments.
- DQN and PPO formulas, pseudocode, flowcharts and the custom network diagrams.
- Exact hyperparameters, device, seeds, episode/timestep totals and wall-clock time.
- Learning curves and independent evaluation mean, standard deviation and random baseline.
- Compare sample efficiency, stability, exploration, replay/off-policy versus on-policy learning.
- Confirm both Atari agents exceed random performance; train beyond one million timesteps if necessary.
- Keep notebook outputs and report numbers identical; include simulation MP4 and AI-use declaration.